<a href="https://colab.research.google.com/github/Sebi2005/Metaheuristics/blob/main/notebooks/Tabu_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving fi10639.tsp to fi10639.tsp
Saving kroC100.tsp to kroC100.tsp


In [ ]:
def load_data_with_capacity(file_name:str):
  with open(file_name) as f:
    lines = [line.strip() for line in f if line.strip()]
  n = int(lines[0])
  items = []
  for i in range(1,n+1):
    parts = lines[i].split()
    weight = int(parts[1])
    value = int(parts[2])
    items.append((weight,value))

  capacity = int(lines[n+1])
  return items,capacity


In [ ]:
def evaluate(solution, items, capacity):
  total_weight = 0
  total_value = 0
  for bit, (weight, value) in zip(solution, items):
    if bit == 1:
      total_weight += weight
      total_value += value
  if total_weight > capacity :
    return 0
  return total_value

def fitness(objects, sol, max_capacity):
    return evaluate(sol, objects, max_capacity)

In [ ]:
import random
def random_solution(n):
  return [random.randint(0,1) for _ in range(n)]

In [ ]:
def is_valid(objects, sol, max_capacity):
    total_weight = 0
    for bit, (weight, value) in zip(sol, objects):
        if bit == 1:
            total_weight += weight
    return total_weight <= max_capacity

In [ ]:
def generate_valid_solution(n, objects, max_capacity):
    sol = [0] * n
    indices = list(range(n))
    random.shuffle(indices)

    current_weight = 0
    for i in indices:
        weight, value = objects[i]
        if current_weight + weight <= max_capacity:
            if random.random() < 0.5:
                sol[i] = 1
                current_weight += weight

    return sol

In [ ]:
def tabu_search_knapsack(max_iterations, tabu_size, objects, max_capacity):

    n = len(objects)

    current = generate_valid_solution(n, objects, max_capacity)
    current_fit = fitness(objects, current, max_capacity)

    best = current[:]
    best_fit = current_fit

    tabu = [0] * n

    for _ in range(max_iterations):


        for i in range(n):
            if tabu[i] > 0:
                tabu[i] -= 1

        best_neighbor = None
        best_neighbor_fit = -1
        best_move = None

        for i in range(n):

            neighbor = current[:]
            neighbor[i] = 1 - neighbor[i]

            if not is_valid(objects, neighbor, max_capacity):
                continue

            fit = fitness(objects, neighbor, max_capacity)


            if tabu[i] == 0 or fit > best_fit:

                if fit > best_neighbor_fit:
                    best_neighbor = neighbor
                    best_neighbor_fit = fit
                    best_move = i

        if best_neighbor is None:
            break

        current = best_neighbor
        current_fit = best_neighbor_fit

        if current_fit > best_fit:
            best = current[:]
            best_fit = current_fit


        tabu[best_move] = tabu_size

    return best, best_fit

In [ ]:
def greedy_TSP(distance_matrix):
    n = len(distance_matrix)

    visited = [False] * n
    route = [0]
    visited[0] = True
    total_cost = 0
    current = 0

    for _ in range(n - 1):
        nearest_city = -1
        min_dist = float("inf")

        for j in range(n):
            if not visited[j] and distance_matrix[current][j] < min_dist:
                min_dist = distance_matrix[current][j]
                nearest_city = j

        route.append(nearest_city)
        visited[nearest_city] = True
        total_cost += min_dist
        current = nearest_city

    total_cost += distance_matrix[current][0]
    route.append(0)

    return route, total_cost

In [ ]:

def distance_matrix_TSP(locations):
    n = len(locations)
    matrix = [[0] * n for _ in range(n)]

    for i in range(n):
        _, x1, y1 = locations[i]
        for j in range(i + 1, n):
            _, x2, y2 = locations[j]
            dist = round(math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2))
            matrix[i][j] = dist
            matrix[j][i] = dist

    return matrix


In [ ]:
items, capacity = load_data_with_capacity("knapsack-20.txt")
best_sol, best_val = tabu_search_knapsack(300, 5, items, capacity)
print(best_sol)
print(best_val)

[0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0]
785


**TSP Resolution using Tabu Search**

**1. Problem Definition**

The Traveling Salesman Problem (TSP) is a classic combinatorial optimization problem. Given a list of cities and the distances between each pair of cities, the objective is to find the shortest possible route that visits each city exactly once and returns to the origin city.

For this assignment, we analyzed two instances:

kroC100: A 100-city problem (Krolak/Felts/Nelson).

fi10639: A large-scale instance containing 10,639 locations in Finland.

In [ ]:
import math
import random
import time


def read_tsp_file(filename):
    locations = []
    reading_coords = False
    with open(filename, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith("NODE_COORD_SECTION"):
                reading_coords = True
                continue
            if line == "EOF" or not line: break
            if reading_coords:
                parts = line.split()
                locations.append((float(parts[1]), float(parts[2])))
    return locations

In [ ]:
def get_dist(c1, c2):
    return math.sqrt((c1[0]-c2[0])**2 + (c1[1]-c2[1])**2)

def calculate_total_distance(route, locations):
    d = 0
    for i in range(len(route)):
        d += get_dist(locations[route[i-1]], locations[route[i]])
    return d

**2. Algorithm Description**

The algorithm implemented is Tabu Search, a metaheuristic that guides a local heuristic search to explore the solution space beyond local optimality.

**Steps / Pseudocode**

Initialization: Generate an initial random permutation of cities. Set this as the Best Solution.

**Neighborhood Exploration:**

Generate a set of candidate moves by swapping two random cities (2-opt move).

Evaluate the total distance of these neighbors.

**Tabu Check:**

Select the best neighbor that is not in the Tabu List.

The Tabu List stores recent moves to prevent the algorithm from cycling back to previously visited states.

**Update:**

Update the Current Solution with the selected neighbor.

Add the move to the Tabu List (removing the oldest move if the list exceeds Tabu Size).

If the new cost is better than the Best Cost, update the Best Solution.

**Termination:**

Repeat for a predefined number of Iterations.

In [ ]:
import random
def tabu_search_tsp(locations, max_iterations, tabu_size):
    n = len(locations)
    current_sol = list(range(n))
    random.shuffle(current_sol)

    best_sol = list(current_sol)
    best_cost = calculate_total_distance(best_sol, locations)

    tabu_list = []

    for _ in range(max_iterations):
        best_neighbor_cost = float('inf')
        best_move = None

        samples = 100
        for _ in range(samples):
            i, j = random.sample(range(n), 2)

            if (i, j) in tabu_list or (j, i) in tabu_list:
                continue


            current_sol[i], current_sol[j] = current_sol[j], current_sol[i]
            cost = calculate_total_distance(current_sol, locations)

            if cost < best_neighbor_cost:
                best_neighbor_cost = cost
                best_move = (i, j)


            current_sol[i], current_sol[j] = current_sol[j], current_sol[i]

        if best_move:
            i, j = best_move
            current_sol[i], current_sol[j] = current_sol[j], current_sol[i]
            tabu_list.append(best_move)
            if len(tabu_list) > tabu_size:
                tabu_list.pop(0)

            if best_neighbor_cost < best_cost:
                best_sol = list(current_sol)
                best_cost = best_neighbor_cost

    return best_cost

In [ ]:
instances = ["kroC100.tsp", "fi10639.tsp"]
results = []


configs = [
    {"iters": 100, "tabu": 10},
    {"iters": 500, "tabu": 20},
    {"iters": 500, "tabu": 50}
]


for inst_name in instances:
    locs = read_tsp_file(inst_name)
    for cfg in configs:
        print(f"Running {inst_name} | Iters: {cfg['iters']} | Tabu: {cfg['tabu']}")
        start_time = time.time()
        run_costs = [tabu_search_tsp(locs, cfg['iters'], cfg['tabu']) for _ in range(3)]
        avg_cost = sum(run_costs) / len(run_costs)

        results.append({
            "Instance": inst_name,
            "Iterations": cfg['iters'],
            "Tabu_Size": cfg['tabu'],
            "Best_Cost": min(run_costs),
            "Avg_Cost": avg_cost,
            "Time_Sec": round(time.time() - start_time, 2)
        })

Starting experiments...
Running kroC100.tsp | Iters: 100 | Tabu: 10
Running kroC100.tsp | Iters: 500 | Tabu: 20
Running kroC100.tsp | Iters: 500 | Tabu: 50
Running fi10639.tsp | Iters: 100 | Tabu: 10
Running fi10639.tsp | Iters: 500 | Tabu: 20
Running fi10639.tsp | Iters: 500 | Tabu: 50


**3. Parameter Settings & Comparative Result**s

Experiments were performed by varying the Number of Iterations and the Tabu Size (Tenure). Each configuration was run 3 times to calculate an average.

|Instance|Iterations|Tabu\_Size|Best\_Cost|Avg\_Cost|Time\_Sec|
|---|---|---|---|---|---|
|kroC100\.tsp|100|10|51937\.343164426566|54485\.64010551529|1\.23|
|kroC100\.tsp|500|20|37690\.95826441094|39312\.27094820687|7\.16|
|kroC100\.tsp|500|50|33142\.02397578866|37977\.10990938091|5\.67|
|fi10639\.tsp|100|10|40986448\.21530881|41187665\.006011076|199\.55|
|fi10639\.tsp|500|20|36047120\.68357152|36125318\.63008242|996\.04|
|fi10639\.tsp|500|50|36007657\.08579224|36211012\.40019025|1000\.31|

In [ ]:
import csv
with open('experiment_results.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

**4. Discussion of Results**

**Impact of Iterations**

For both instances, increasing the iterations from 100 to 500 resulted in a massive improvement in solution quality. In kroC100, the best cost dropped by approximately 36%. This confirms that Tabu Search requires a sustained search period to escape initial random configurations.

**Impact of Tabu Size**

A larger Tabu Size (increasing from 20 to 50 at 500 iterations) led to the best overall results for both instances.

In kroC100, the best cost improved from 37,690 to 33,142.

A larger tabu size forces the algorithm to explore new regions of the search space by blocking more "recent" moves, effectively preventing the search from stagnating in local minima.

**Instance Scale & Performance**

kroC100: The algorithm is highly efficient, finding significantly better routes in under 8 seconds.

fi10639: Due to the massive dimensionality (10k+ cities), the time complexity increases drastically. While the cost improved with more iterations, the execution time jumped to ~1000 seconds. For a problem this large, the "Best Cost" found is likely still far from the global optimum

**Conclusion**

Tabu Search proved effective at refining random TSP routes. The experiments show that longer search times (Iterations) combined with a robust memory (Tabu Size) are critical for achieving high-quality solutions in combinatorial optimization tasks.